<div style="background-color: #161b22; padding: 20px; border-radius: 12px; border-left: 6px solid #2ecc71; box-shadow: 0 4px 6px rgba(0,0,0,0.3);">
  <h1 style="color: #ffffff; margin: 0; font-family: sans-serif; font-weight: 700; letter-spacing: 1px;">
    ⚡ Z-Image Turbo <span style="font-size: 0.6em; color: #2ecc71; vertical-align: middle; background: #2ecc7122; padding: 2px 8px; border-radius: 6px;">Multi-LoRA Support (4 Slots)</span>
  </h1>
  <p style="color: #8b949e; margin: 8px 0 0 0; font-family: sans-serif;">
    • Hybrid FP8/GGUF + LoRA Optimized
  </p>
</div>

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**
*Subscribe for more AI tutorials, workflows, and optimization tips!*

In [ ]:
#@title 1. Initialize Core Environment
#@markdown This prepares the ephemeral storage, installs ComfyUI, and configures the GGUF integration tools.

import os
import subprocess

LOCAL_WORKSPACE = "/content/ComfyUI"

print("🚀 Initializing Core Architecture...")
if not os.path.exists(LOCAL_WORKSPACE):
    !git clone https://github.com/comfyanonymous/ComfyUI {LOCAL_WORKSPACE} &> /dev/null
    print("   ✓ Core Engine Cloned")
else:
    !cd {LOCAL_WORKSPACE} && git pull &> /dev/null
    print("   ✓ Core Engine Updated")

print("📦 Installing Dependencies (This takes a moment)...")
!cd {LOCAL_WORKSPACE} && pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 &> /dev/null

GGUF_NODE_DIR = os.path.join(LOCAL_WORKSPACE, "custom_nodes/ComfyUI-GGUF")
if not os.path.exists(GGUF_NODE_DIR):
    print("🧩 Installing GGUF Processing Nodes...")
    !git clone https://github.com/city96/ComfyUI-GGUF {GGUF_NODE_DIR} &> /dev/null
    !pip install -r {GGUF_NODE_DIR}/requirements.txt &> /dev/null

print("✅ Environment Ready!")

In [ ]:
#@title 2. High-Speed Asset Downloader
#@markdown Paste your HuggingFace/Civitai model links here. **You can paste multiple LoRA URLs separated by commas or new lines** — they will all be downloaded to the loras folder. The required Z-Image base assets (Qwen Text Encoder & VAE) are automatically fetched.

import os
import subprocess
import gdown
import urllib.parse

WORKSPACE = "/content/ComfyUI"

# --- Input Resources ---
UNET_URLS = "" #@param {type:"string"}
LORA_URLS = "" #@param {type:"string"}
#@markdown *Optional: needed for Civitai models that require login (401/403 errors)*
CIVITAI_API_TOKEN = "" #@param {type:"string"}

# Pre-configured required models
TEXT_ENCODER_URLS = "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors"
VAE_URLS = "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors"

DIRS = {
    "unet":           os.path.join(WORKSPACE, "models/unet"),
    "clip":           os.path.join(WORKSPACE, "models/clip"),
    "vae":            os.path.join(WORKSPACE, "models/vae"),
    "loras":          os.path.join(WORKSPACE, "models/loras"),
}

print("⚡ Configuring Aria2c Accelerator...")
subprocess.run(['apt-get', '-y', 'install', '-qq', 'aria2'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def download_file(url, target_dir):
    try:
        os.makedirs(target_dir, exist_ok=True)
        before_files = set(os.listdir(target_dir))

        if "drive.google.com" in url:
            print(f"   📥 Downloading from Drive...")
            gdown.download(url, output=target_dir + '/', quiet=False, fuzzy=True)
        else:
            print(f"   📥 Fetching: {url.split('/')[-1][:40]}...")
            parsed_url = urllib.parse.urlparse(url)
            filename = os.path.basename(parsed_url.path)

            aria2_cmd = [
                "aria2c", "--console-log-level=error", "--summary-interval=10",
                "-c", "-x", "16", "-s", "16", "-k", "1M"
            ]

            is_civitai = "civitai" in parsed_url.netloc.lower()
            has_extension = os.path.splitext(filename)[1].lower() in (
                ".safetensors", ".gguf", ".ckpt", ".pt", ".pth", ".bin", ".sft", ".zip")

            if is_civitai and "/api/download/" in parsed_url.path and CIVITAI_API_TOKEN.strip():
                sep = "&" if parsed_url.query else "?"
                url = f"{url}{sep}token={CIVITAI_API_TOKEN.strip()}"

            if is_civitai or not has_extension:
                # Let the server name the file (follows redirects, reads Content-Disposition)
                aria2_cmd.extend(["--content-disposition", url, "-d", target_dir])
            else:
                aria2_cmd.extend(["-o", filename, url, "-d", target_dir])

            subprocess.run(aria2_cmd, check=True)

        after_files = set(os.listdir(target_dir))
        new_files = after_files - before_files

        if new_files:
            downloaded_file = list(new_files)[0]
            print(f"   ✅ Saved as: \033[96m{downloaded_file}\033[0m")
        else:
            print(f"   ⏭️ Already in library — skipped (existing file kept)")

    except Exception as e:
        print(f"   ❌ Failed: {url}\n      Error: {e}\n")

def process_downloads(urls_str, target_dir):
    if not urls_str.strip(): return
    url_list = [u.strip() for u in urls_str.replace(',', '\n').split('\n') if u.strip()]
    os.makedirs(target_dir, exist_ok=True)
    print(f"\n📂 Directory: {os.path.basename(target_dir)}")
    for url in url_list:
        download_file(url, target_dir)

process_downloads(UNET_URLS,         DIRS["unet"])
process_downloads(LORA_URLS,         DIRS["loras"])
process_downloads(TEXT_ENCODER_URLS, DIRS["clip"])
process_downloads(VAE_URLS,          DIRS["vae"])

print("\n✅ All assets secured!")

# --- Asset Library Inventory ---
def _human_size(num_bytes):
    for unit in ["B", "KB", "MB", "GB"]:
        if num_bytes < 1024 or unit == "GB":
            return f"{num_bytes:.1f} {unit}" if unit != "B" else f"{num_bytes} B"
        num_bytes /= 1024

import html as _htmlmod
import json as _jsonmod
from IPython.display import display as _display, HTML as _HTML

_inv = ['<div style="background:#161b22;border:1px solid #30363d;border-radius:10px;'
        'padding:14px 18px;font-family:monospace;margin:8px 0;max-width:640px;">'
        '<div style="color:#2ecc71;font-weight:bold;font-size:14px;">📋 ASSET LIBRARY</div>'
        '<div style="color:#8b949e;font-size:11px;margin-bottom:8px;">'
        'Click any name to copy it, then paste into the Generation cell</div>']
for label, key in [("🧠 UNet Models → UNET_FILENAME", "unet"),
                   ("🎨 LoRAs → LORA_1..4_FILENAME", "loras")]:
    folder = DIRS[key]
    entries = sorted(f for f in os.listdir(folder)) if os.path.exists(folder) else []
    entries = [f for f in entries if not f.endswith(".aria2")]
    _inv.append(f'<div style="color:#e6edf3;font-size:12px;margin:10px 0 4px 0;">{label}:</div>')
    if not entries:
        _inv.append('<div style="color:#8b949e;font-size:12px;margin-left:12px;">(empty)</div>')
    for f in entries:
        size = _human_size(os.path.getsize(os.path.join(folder, f)))
        safe_html = _htmlmod.escape(f)
        safe_js = _htmlmod.escape(_jsonmod.dumps(f), quote=True)
        _inv.append(
            f'<div style="margin:3px 0 3px 12px;">'
            f'<code onclick="navigator.clipboard.writeText({safe_js});'
            f'var b=this.nextElementSibling;b.textContent=\'✓ copied!\';'
            f'setTimeout(function(){{b.textContent=\'({size})\';}},1500);" '
            f'style="color:#58d6ff;background:#0d1117;border:1px solid #30363d;'
            f'border-radius:6px;padding:3px 8px;cursor:pointer;font-size:13px;" '
            f'title="Click to copy">{safe_html}</code> '
            f'<span style="color:#8b949e;font-size:11px;">({size})</span></div>')
_inv.append('</div>')
_display(_HTML("".join(_inv)))

In [ ]:
#@title 3. Image Generation
#@markdown Enter your prompt and tweak parameters.

# --- Model Selection ---
UNET_FILENAME = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 🎨 LoRA Stack — a slot is skipped if its filename is empty/"none" **or its strength is 0** (easy on/off without retyping names)
LORA_1_FILENAME = "" #@param {type:"string"}
LORA_1_STRENGTH = 1.0 #@param {type:"slider", min:0.0, max:2.0, step:0.05}
LORA_2_FILENAME = "" #@param {type:"string"}
LORA_2_STRENGTH = 1.0 #@param {type:"slider", min:0.0, max:2.0, step:0.05}
LORA_3_FILENAME = "" #@param {type:"string"}
LORA_3_STRENGTH = 1.0 #@param {type:"slider", min:0.0, max:2.0, step:0.05}
LORA_4_FILENAME = "" #@param {type:"string"}
LORA_4_STRENGTH = 1.0 #@param {type:"slider", min:0.0, max:2.0, step:0.05}
#@markdown ---

# --- Generation Settings ---
PROMPT = "" #@param {type:"string"}
NEGATIVE_PROMPT = "blurry, low quality, deformed, artifacts" #@param {type:"string"}

#@markdown ---
#@markdown ### 📐 Resolution — every option shows its exact size, pick and done
RESOLUTION = "16:9 · FHD · 1920x1080" #@param ["Custom (use sliders below)", "1:1 · SD · 720x720", "1:1 · HD · 896x896", "1:1 · FHD · 1080x1080", "16:9 · SD · 1280x720", "16:9 · HD · 1600x896", "16:9 · FHD · 1920x1080", "9:16 · SD · 720x1280", "9:16 · HD · 896x1600", "9:16 · FHD · 1080x1920", "4:3 · SD · 960x720", "4:3 · HD · 1184x888", "4:3 · FHD · 1440x1080", "3:4 · SD · 720x960", "3:4 · HD · 888x1184", "3:4 · FHD · 1080x1440", "3:2 · SD · 1080x720", "3:2 · HD · 1344x896", "3:2 · FHD · 1632x1088", "2:3 · SD · 720x1080", "2:3 · HD · 896x1344", "2:3 · FHD · 1088x1632", "21:9 · SD · 1344x576", "21:9 · HD · 1680x720", "21:9 · FHD · 2016x864"]
#@markdown *Custom size (only used when RESOLUTION is "Custom"): use the sliders, or type exact numbers in the override boxes (0 = use slider)*
WIDTH = 1024 #@param {type:"slider", min:512, max:2048, step:8}
HEIGHT = 1024 #@param {type:"slider", min:512, max:2048, step:8}
WIDTH_OVERRIDE = 0 #@param {type:"integer"}
HEIGHT_OVERRIDE = 0 #@param {type:"integer"}
#@markdown ---
BATCH_SIZE = 1 #@param {type:"slider", min:1, max:4, step:1}

from IPython.display import display as _display, HTML as _HTML

if "Custom" not in RESOLUTION:
    ratio_key, _quality, _dims = [p.strip() for p in RESOLUTION.split("·")]
    WIDTH, HEIGHT = (int(v) for v in _dims.split("x"))
    _res_source = f"Preset: {ratio_key} @ {_quality} — sliders are IGNORED"
else:
    ratio_key = "custom"
    WIDTH = int(WIDTH_OVERRIDE) if int(WIDTH_OVERRIDE) > 0 else int(WIDTH)
    HEIGHT = int(HEIGHT_OVERRIDE) if int(HEIGHT_OVERRIDE) > 0 else int(HEIGHT)
    WIDTH, HEIGHT = max(64, WIDTH), max(64, HEIGHT)
    if WIDTH % 8 or HEIGHT % 8:
        WIDTH, HEIGHT = (WIDTH // 8) * 8, (HEIGHT // 8) * 8
        _res_source = "Custom sliders (snapped to nearest multiple of 8)"
    else:
        _res_source = "Custom sliders"

# --- Visual resolution confirmation (this is EXACTLY what the model receives) ---
_max_box = 220
_scale = _max_box / max(WIDTH, HEIGHT)
_bw, _bh = int(WIDTH * _scale), int(HEIGHT * _scale)
_display(_HTML(f"""
<div style="background:#161b22;border:1px solid #2ecc71;border-radius:10px;padding:14px 18px;
            font-family:monospace;display:inline-block;margin:6px 0;">
  <div style="color:#2ecc71;font-size:15px;font-weight:bold;">
    📐 FINAL CANVAS → {WIDTH} × {HEIGHT} px
  </div>
  <div style="color:#8b949e;font-size:12px;margin:2px 0 10px 0;">{_res_source}</div>
  <div style="width:{_bw}px;height:{_bh}px;background:linear-gradient(135deg,#1f6feb33,#2ecc7133);
              border:2px dashed #2ecc71;border-radius:4px;display:flex;
              align-items:center;justify-content:center;color:#e6edf3;font-size:12px;">
    {ratio_key if ratio_key != "custom" else f"{WIDTH}x{HEIGHT}"}
  </div>
</div>
"""))

# --- Advanced Sampler Settings ---
STEPS = 9 #@param {type:"slider", min:1, max:20, step:1}
CFG = 1.0 #@param {type:"number"}
SAMPLER_NAME = "res_multistep" #@param ["euler", "euler_cfg_pp", "euler_ancestral", "euler_ancestral_cfg_pp", "heun", "heunpp2", "dpm_2", "dpm_2_ancestral", "lms", "dpm_fast", "dpm_adaptive", "dpmpp_2s_ancestral", "dpmpp_2s_ancestral_cfg_pp", "dpmpp_sde", "dpmpp_sde_gpu", "dpmpp_2m", "dpmpp_2m_cfg_pp", "dpmpp_2m_sde", "dpmpp_2m_sde_gpu", "dpmpp_3m_sde", "dpmpp_3m_sde_gpu", "ddpm", "lcm", "ipndm", "ipndm_v", "deis", "res_multistep", "res_multistep_cfg_pp", "res_multistep_ancestral", "res_multistep_ancestral_cfg_pp", "gradient_estimation", "gradient_estimation_cfg_pp", "er_sde", "seeds_2", "seeds_3", "sa_solver", "sa_solver_pece", "ddim", "uni_pc", "uni_pc_bh2"] {allow-input: true}
SCHEDULER = "beta" #@param ["normal", "karras", "exponential", "sgm_uniform", "simple", "ddim_uniform", "beta", "linear_quadratic", "kl_optimal"] {allow-input: true}
AURA_SHIFT = 3.0 #@param {type:"slider", min:1.0, max:10.0, step:0.5}
SEED = 0 #@param {type:"integer"}

import sys
import os
import json
import time
import random
import subprocess
import urllib.request
from IPython.display import display, Image as IPImage

WORKSPACE = "/content/ComfyUI"
os.chdir(WORKSPACE)

if SEED == 0:
    SEED = random.randint(1, 1125899906842624)

print("🔌 Checking ComfyUI Server Status...")
def start_server():
    req = urllib.request.Request("http://127.0.0.1:8188")
    try:
        urllib.request.urlopen(req)
        print("   🟢 Server is already running.")
    except:
        print("   🚀 Starting ComfyUI Server in background...")
        subprocess.Popen([sys.executable, "main.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        while True:
            try:
                urllib.request.urlopen(req)
                print("   🟢 Server is now up and ready!")
                break
            except:
                time.sleep(2)

start_server()
print(f"\033[94m➜ Generation Details | Size: {WIDTH}x{HEIGHT} ({ratio_key}) | Seed: {SEED}\033[0m")

prompt_workflow = {
    "9": {"inputs": {"filename_prefix": "z-image", "images": ["43", 0]}, "class_type": "SaveImage"},
    "39": {"inputs": {"clip_name": "qwen_3_4b.safetensors", "type": "lumina2", "device": "default"}, "class_type": "CLIPLoader"},
    "40": {"inputs": {"vae_name": "ae.safetensors"}, "class_type": "VAELoader"},
    "41": {"inputs": {"width": WIDTH, "height": HEIGHT, "batch_size": BATCH_SIZE}, "class_type": "EmptySD3LatentImage"},
    "42": {"inputs": {"text": NEGATIVE_PROMPT, "clip": ["39", 0]}, "class_type": "CLIPTextEncode"},
    "43": {"inputs": {"samples": ["44", 0], "vae": ["40", 0]}, "class_type": "VAEDecode"},
    "44": {"inputs": {"seed": SEED, "steps": STEPS, "cfg": CFG, "sampler_name": SAMPLER_NAME, "scheduler": SCHEDULER, "denoise": 1, "model": ["47", 0], "positive": ["45", 0], "negative": ["42", 0], "latent_image": ["41", 0]}, "class_type": "KSampler"},
    "45": {"inputs": {"text": PROMPT, "clip": ["39", 0]}, "class_type": "CLIPTextEncode"},
    "47": {"inputs": {"shift": AURA_SHIFT, "model": ["48", 0]}, "class_type": "ModelSamplingAuraFlow"}
}

# Smart UNet Node Routing based on file extension
if UNET_FILENAME.lower().endswith('.gguf'):
    prompt_workflow["48"] = {"inputs": {"unet_name": UNET_FILENAME}, "class_type": "UnetLoaderGGUF"}
else:
    prompt_workflow["48"] = {"inputs": {"unet_name": UNET_FILENAME, "weight_dtype": "default"}, "class_type": "UNETLoader"}

# Multi-LoRA Chain Routing
# Each active LoRA slot becomes a LoraLoaderModelOnly node, chained in order:
# UNet -> LoRA 1 -> LoRA 2 -> LoRA 3 -> LoRA 4 -> ModelSamplingAuraFlow
LORA_SLOTS = [
    (LORA_1_FILENAME, LORA_1_STRENGTH),
    (LORA_2_FILENAME, LORA_2_STRENGTH),
    (LORA_3_FILENAME, LORA_3_STRENGTH),
    (LORA_4_FILENAME, LORA_4_STRENGTH),
]
active_loras = [(name.strip(), strength) for name, strength in LORA_SLOTS
                if name.strip() and name.strip().lower() != "none" and strength != 0]
disabled_loras = [name.strip() for name, strength in LORA_SLOTS
                  if name.strip() and name.strip().lower() != "none" and strength == 0]

prev_model_link = ["48", 0]
for idx, (lora_name, lora_strength) in enumerate(active_loras):
    node_id = str(50 + idx)
    prompt_workflow[node_id] = {
        "inputs": {"lora_name": lora_name, "strength_model": lora_strength, "model": prev_model_link},
        "class_type": "LoraLoaderModelOnly"
    }
    prev_model_link = [node_id, 0]
prompt_workflow["47"]["inputs"]["model"] = prev_model_link

if active_loras:
    print("\033[95m➜ Active LoRA Stack:\033[0m")
    for i, (lora_name, lora_strength) in enumerate(active_loras, 1):
        print(f"   {i}. {lora_name}  (strength: {lora_strength})")
else:
    print("\033[93m➜ No LoRAs active — running base model only.\033[0m")
if disabled_loras:
    for lora_name in disabled_loras:
        print(f"\033[90m   ⏸ Disabled (strength 0): {lora_name}\033[0m")

p = {"prompt": prompt_workflow}
data = json.dumps(p).encode('utf-8')
req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)

print("   📥 Submitting workflow to API...")
try:
    response = urllib.request.urlopen(req)
    prompt_id = json.loads(response.read())['prompt_id']
except Exception as e:
    print(f"   ❌ API Error: {e}")
    raise

print("   ✨ Processing and Sampling (Check ComfyUI server logs if stuck)...")
while True:
    try:
        history_req = urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}")
        history_res = urllib.request.urlopen(history_req)
        history_data = json.loads(history_res.read())
        if prompt_id in history_data:
            outputs = history_data[prompt_id]['outputs']
            break
    except:
        pass
    time.sleep(1)

print("   🖼️ Decoding Final Masterpiece...")
for node_id, node_output in outputs.items():
    if 'images' in node_output:
        for image in node_output['images']:
            filename = image['filename']
            img_path = os.path.join(WORKSPACE, "output", filename)
            print(f"\033[92m✓ Saved: {filename}\033[0m")
            display(IPImage(filename=img_path))

In [ ]:
 #@title 4. Export & Download Results
#@markdown Run this to instantly zip and download all the generated images from this session.

import os
from google.colab import files

OUTPUT_DIR = "/content/ComfyUI/output"
ZIP_NAME = "/content/Z_Image_Artworks.zip"

if os.path.exists(OUTPUT_DIR) and len(os.listdir(OUTPUT_DIR)) > 0:
    print("🗜️ Zipping generated artworks...")
    !zip -j -q {ZIP_NAME} {OUTPUT_DIR}/*.png
    print("📥 Initiating download...")
    files.download(ZIP_NAME)
else:
    print("⚠️ No images found in the output directory yet!")